In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Complete a form and confirm before submitting

| | |
|-|-|
| Author(s) | [Matt Robinson](https://github.com/mr394729) |

> **This copy keeps the output of one complete run** (24 September 2026, in a test namespace), so you can read what each cell prints even if a cell fails for you. Your numbers and wording will differ where a model answers. To start clean, choose **Edit > Clear Outputs of All Cells** in JupyterLab, or run `jupyter nbconvert --clear-output --inplace <notebook>`.

## Overview

### Collecting a form in conversation

Many store tasks are forms: a damage report, a price change, a transfer request. An agent can fill the form from what the person already said and ask only for the missing fields, one question at a time. Code, not the model, checks every value before anything is submitted.

### Tool confirmation

ADK can pause before a tool runs and ask the user to approve it. Wrap the function in `FunctionTool(..., require_confirmation=True)`: when the model calls it, ADK stops the turn and sends a confirmation request instead. The tool runs only after the user approves. See [tool confirmation](https://google.github.io/adk-docs/tools/confirmation/).

### The incident report

An associate reports damaged, missing or suspicious units for the shrink log. The report has five fields: product, quantity, event type (`damage`, `unknown_loss` or `return_anomaly`), location in the store, and a short note. In this quickstart the submit step returns the record it would log and writes nothing.

<img width="60%" src="../../docs/diagrams/q03.png" alt="An agent that fills an incident report field by field and submits it after confirmation" />

### Objectives

In this tutorial, you will learn how to collect a form in conversation and require the user's approval before a tool runs.

You will complete the following tasks:

- Write a tool that finds a product in the catalog
- Write a submit tool that validates every field and requires confirmation
- Run the agent until it asks for approval, then approve it

### Costs

This tutorial uses billable components of Google Cloud:

- Gemini on Vertex AI
- BigQuery

Learn about [Gemini on Vertex AI pricing](https://cloud.google.com/vertex-ai/generative-ai/pricing), [BigQuery pricing](https://cloud.google.com/bigquery/pricing), and use the [Pricing Calculator](https://cloud.google.com/products/calculator/) to generate a cost estimate based on your projected usage.

## Get started

### Set Google Cloud project information

This quickstart reads the store data you loaded during setup, in your own namespace. Set your project ID and the namespace you chose.

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [2]:
import os
import sys
from pathlib import Path

PROJECT_ID = "[your-project-id]"  # @param {type: "string"}
WORKSHOP_NAMESPACE = "[your-namespace]"  # @param {type: "string"}

if PROJECT_ID == "[your-project-id]":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "")
if WORKSHOP_NAMESPACE == "[your-namespace]":
    WORKSHOP_NAMESPACE = os.environ.get("WORKSHOP_NAMESPACE", "")
if not PROJECT_ID or not WORKSHOP_NAMESPACE:
    raise ValueError("Set PROJECT_ID and WORKSHOP_NAMESPACE above.")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["WORKSHOP_NAMESPACE"] = WORKSHOP_NAMESPACE
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["STORE_OPS_ENV"] = "dev"

# This notebook sits two folders below the repository root, where the shared store tools live
REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT))

### Import libraries

In [3]:
import logging
import warnings

# Keep the notebook output to the agent's own events: ADK marks experimental features with a
# UserWarning, SDKs announce renamed classes with a FutureWarning, and the Gen AI SDK logs a note
# whenever a response mixes text and tool calls.
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("google_genai").setLevel(logging.ERROR)

import hashlib
import re

from google.adk.agents import LlmAgent
from google.adk.models import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import FunctionTool, ToolContext
from google.genai import types

from agents.cymbal_store_ops.tools.domain_tools import search_products

### Choose the model

The agent in this tutorial uses Gemini 3.8 Flash. The retry options make the SDK retry a request that fails with a temporary error, such as a 429 or a 500, instead of failing the turn.

In [4]:
model = Gemini(
    model="gemini-3.8-flash",
    retry_options=types.HttpRetryOptions(attempts=4, initial_delay=2.0),
)

## Define the tools

### Find the product

The associate names a product the way it reads on the shelf, for example "Hydra Cream". `find_product` looks it up in the catalog with the shared `search_products` tool and returns the product IDs that match, so the model never has to guess an ID.

In [5]:
def find_product(product_name: str) -> dict:
    """Resolve a product the associate named (e.g. "Hydra Cream") to catalog rows: product_id, name, brand, category."""
    result = search_products(query_text=product_name, limit=5)
    if result.get("status") != "SUCCESS":
        return result
    if not result["rows"]:
        return {"status": "ERROR", "error_details": f"no product matches {product_name!r}"}
    keep = ("product_id", "name", "brand", "category")
    return {"status": "SUCCESS", "rows": [{key: row[key] for key in keep} for row in result["rows"]]}


find_product("Hydra Cream")

{'status': 'SUCCESS',
 'rows': [{'product_id': 'P-0101',
   'name': 'Lumière Hydra Cream',
   'brand': 'Lumière Skin',
   'category': 'skincare'}]}

One product matches: `P-0101`, Lumière Hydra Cream.

### Submit the report

`submit_incident_report` checks every field in code: the product ID format, a quantity between 1 and 50, one of the three event types, a location, and a one-sentence note. It reads the store from the session state, so the associate never types it. It returns the record with `written: False`, because this quickstart does not write to the shrink log.

In [6]:
EVENT_TYPES = ["damage", "unknown_loss", "return_anomaly"]


def submit_incident_report(
    product_id: str, quantity: int, event_type: str, location: str, note: str, tool_context: ToolContext
) -> dict:
    """Submit the incident report after the associate confirms.

    Args:
        product_id: from find_product, e.g. P-0101.
        quantity: units affected (1-50).
        event_type: damage | unknown_loss | return_anomaly.
        location: where in the store, e.g. "skincare aisle" or "backroom".
        note: one sentence about what happened; no names or contact details.
    """
    problems = []
    if not re.fullmatch(r"P-\d{4}", product_id or ""):
        problems.append("product_id must come from find_product (like P-0101)")
    if not 1 <= int(quantity) <= 50:
        problems.append("quantity must be 1-50")
    if event_type not in EVENT_TYPES:
        problems.append(f"event_type must be one of {EVENT_TYPES}")
    if not location.strip():
        problems.append("location is required")
    if not note.strip() or len(note) > 200:
        problems.append("note must be one sentence")
    if problems:
        return {"status": "ERROR", "error_details": "; ".join(problems)}

    record = {
        "store_id": tool_context.state["user:store_id"],
        "product_id": product_id,
        "quantity": int(quantity),
        "event_type": event_type,
        "location": location.strip(),
        "note": note.strip(),
    }
    incident_id = "INC-" + hashlib.sha256(repr(sorted(record.items())).encode()).hexdigest()[:6].upper()
    return {"status": "SUCCESS", "written": False, "incident_id": incident_id, "record": record}

## Define the agent

Register the submit tool with `require_confirmation=True`. The instruction reads the signed-in store from the session state with `{user:store_id}`.

In [7]:
instruction = f"""You take incident reports from Cymbal Beauty store associates for the shrink log.
Signed-in store: {{user:store_id}}. A report has five fields: product, quantity, event type
({", ".join(EVENT_TYPES)}), location in the store, and a short note.
- Use everything the associate already said; ask only for what is missing, one short question at a time.
- As soon as you know the product, call find_product; if several products match, ask which one.
- The note says what happened, never who.
- When all five fields are known, call submit_incident_report once. Then give the incident id and say
  the report was not written to the shrink log, because this is a quickstart.
Never invent a product id, a quantity or a location."""

agent = LlmAgent(
    name="form_completion_agent",
    model=model,
    description="Takes an associate's damage or loss report field by field and submits it after confirmation.",
    instruction=instruction,
    tools=[find_product, FunctionTool(submit_incident_report, require_confirmation=True)],
)

## Run the agent

Start a session as Priya, an associate at store S-014:

In [8]:
runner = InMemoryRunner(agent=agent, app_name="form_completion_agent")
session = await runner.session_service.create_session(
    app_name="form_completion_agent",
    user_id="priya",
    state={"user:user_id": "A-1004", "user:store_id": "S-014", "user:role": "associate"},
)

Define a helper that sends a message and prints what happened. It also keeps any confirmation request, because answering one needs its ID.

In [9]:
pending_confirmation = None


async def send(message: types.Content) -> None:
    """Send one message and print tool calls, confirmation requests and the final answer."""
    global pending_confirmation
    async for event in runner.run_async(
        user_id=session.user_id, session_id=session.id, new_message=message
    ):
        for call in event.get_function_calls():
            if call.name == "adk_request_confirmation":
                pending_confirmation = call
                original = call.args["originalFunctionCall"]
                print(f"Confirmation requested for {original['name']}({original['args']})")
            else:
                print(f"[{event.author}] calls {call.name}({call.args})")
        if event.is_final_response() and event.content and event.content.parts:
            text = "".join(part.text or "" for part in event.content.parts if not part.thought)
            if text:
                print(f"\n{text}\n")


async def ask(question: str) -> None:
    await send(types.Content(role="user", parts=[types.Part(text=question)]))

Report damage without the quantity. The agent looks up the product and asks only for what is missing:

In [10]:
await ask("Log damage for Lumière Hydra Cream in the skincare aisle: the jars were crushed in a delivery tote.")

[form_completion_agent] calls find_product({'product_name': 'Lumière Hydra Cream'})



How many jars were damaged?



The message gave the product, the event type, the location and what happened. The agent resolved the product to `P-0101` and asked only for the quantity. The wording of the question varies from run to run.

Give the quantity. Now all five fields are known, so the model calls the submit tool, and ADK stops the turn with a confirmation request:

In [11]:
await ask("2 units")

[form_completion_agent] calls submit_incident_report({'quantity': 2, 'event_type': 'damage', 'location': 'skincare aisle', 'product_id': 'P-0101', 'note': 'The jars were crushed in a delivery tote.'})
Confirmation requested for submit_incident_report({'quantity': 2, 'event_type': 'damage', 'location': 'skincare aisle', 'product_id': 'P-0101', 'note': 'The jars were crushed in a delivery tote.'})


The tool has not run yet. The first line is the model's call; ADK held it back, sent an `adk_request_confirmation` request instead, and ended the turn. The helper kept that request in `pending_confirmation`. The agent took the note from the associate's own words.

### Approve the submission

In an app, the confirmation request is a dialog with Approve and Reject. Here you answer it in code: send a function response for `adk_request_confirmation` with the request's ID and `confirmed: True`. ADK then runs the tool and the agent reports the result.

In [12]:
approval = types.Content(
    role="user",
    parts=[
        types.Part(
            function_response=types.FunctionResponse(
                id=pending_confirmation.id,
                name="adk_request_confirmation",
                response={"confirmed": True},
            )
        )
    ],
)
await send(approval)


The incident report has been submitted under incident ID **INC-FE8BD7**. 

Please note that the report was not written to the shrink log, because this is a quickstart.



ADK ran `submit_incident_report` and the agent gave the incident ID, `INC-FE8BD7` in this run. The ID is a hash of the record, so it changes when the model words the note differently.

Try these example phrases in a new session, and reject the confirmation with `confirmed: False` to see the agent stop:

```
Three bottles of Noir Velvet Eau de Parfum are missing from the locked case
A guest returned an opened concealer that looks swapped
```

## Run the agent in the ADK developer UI

`agent.py` in this folder defines the same agent, plus `get_user_choice`, which the developer UI shows as buttons for the event type, and `identify_demo_user`, which signs you in by ID. From the repository root:

```bash
uv run python scripts/quickstart_apps.py 03-form-completion-agent
uv run adk web build/quickstart_apps --port 8001
```

Choose `qs_03_form_completion_agent` and start with "I'm A-1004". The confirmation appears as a dialog.

## Cleaning up

This notebook creates no cloud resources and writes no records.

## What's next

- [Tool confirmation in ADK](https://google.github.io/adk-docs/tools/confirmation/)
- [Session state](https://google.github.io/adk-docs/sessions/state/)
- [Quickstart 04: call external APIs](../04-external-api-agent/walkthrough.ipynb)